## Import packages

In [1]:
# Needed this for loading the dataset locally
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from datasets import load_dataset
import numpy as np


import nltk

In [2]:
# dataset = load_dataset("coastalcph/tydi_xor_rc")

## Load Dataset

In [3]:
dataset = load_dataset("coastalcph/tydi_xor_rc")

## Filter and split Dataset

In [4]:
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()



#filter data 
langlst = ['ko','ar','te']
# Source - https://stackoverflow.com/a/59275490
# Posted by atinjanki
# Retrieved 2026-09-05, License - CC BY-SA 4.0

df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#filter data 
#Training data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

#Validation data for each language
df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]



In [5]:
# --------------------------- What is this for? --------------------------- 

# df_train_ar.duplicated().value_counts()
# df_train_ko.duplicated().value_counts()
# df_train_te.duplicated().value_counts()

print(f"{df_train_ar.duplicated().value_counts()}")
print(f"{df_train_ko.duplicated().value_counts()}")
print(f"{df_train_te.duplicated().value_counts()}")

# --------------------------- What is this for? --------------------------- 

False    2558
Name: count, dtype: int64
False    2412
True       10
Name: count, dtype: int64
False    1355
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")



def get_statistics(df, split, language):
    question_lengths = []
    context_lengths = []

    for question in df["question"]:
        question_lengths.append(len(tokenizer.tokenize(question)))
    
    for context in df["context"]:
        context_lengths.append(len(tokenizer.tokenize(context)))

    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])


    return {
            "split": split,
            "language": language,
            "n": len(df),
            "answerable_%": 100 * answerable_n / len(df),
            "unanswerable_%": 100 * unanswerable_n / len(df),
            "question_median": np.median(question_lengths),
            "question_IQR": np.percentile(question_lengths, 75) - np.percentile(question_lengths, 25),
            "context_median": np.median(context_lengths),
            "context_IQR": np.percentile(context_lengths, 75) - np.percentile(context_lengths, 25)
        }


statistics = pd.DataFrame([
    get_statistics(df_train_ar, "train", "ar"),
    get_statistics(df_train_ko, "train", "ko"),
    get_statistics(df_train_te, "train", "te"),
    get_statistics(df_validation_ar, "validation", "ar"),
    get_statistics(df_validation_ko, "validation", "ko"),
    get_statistics(df_validation_te, "validation", "te")
])

statistics.round(1)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors


[163, 110, 51, 151, 220, 416, 130, 186, 111, 157, 150, 163, 64, 122, 120, 209, 250, 200, 64, 117, 241, 137, 551, 236, 167, 144, 142, 141, 129, 117, 129, 88, 161, 70, 127, 106, 92, 102, 96, 102, 100, 76, 171, 124, 267, 106, 65, 92, 108, 174, 70, 201, 92, 44, 219, 307, 87, 163, 65, 62, 88, 197, 125, 109, 157, 185, 141, 265, 278, 106, 92, 105, 146, 48, 91, 128, 199, 24, 41, 53, 233, 230, 38, 50, 139, 102, 261, 93, 57, 102, 84, 125, 139, 87, 146, 148, 316, 214, 77, 320, 307, 155, 101, 120, 142, 180, 162, 125, 132, 150, 50, 189, 79, 155, 79, 44, 23, 272, 145, 152, 213, 44, 28, 163, 208, 75, 67, 21, 125, 176, 190, 117, 60, 175, 49, 406, 209, 217, 115, 34, 157, 157, 92, 254, 88, 24, 56, 55, 122, 131, 48, 288, 166, 206, 75, 210, 38, 150, 193, 184, 117, 73, 76, 316, 116, 108, 98, 88, 93, 37, 63, 88, 170, 97, 216, 300, 79, 245, 174, 52, 238, 45, 224, 548, 112, 53, 135, 166, 126, 127, 68, 214, 333, 137, 161, 90, 133, 107, 107, 161, 141, 76, 149, 88, 61, 218, 203, 143, 225, 204, 166, 88, 119, 111,

,split,language,n,answerable_%,unanswerable_%,question_median,question_IQR,context_median,context_IQR
0,train,ar,2558,90.0,10.0,13.0,6.0,123.0,94.0
1,train,ko,2422,97.4,2.6,14.0,4.0,115.0,87.0
2,train,te,1355,96.7,3.3,18.0,6.5,114.0,87.0
3,validation,ar,415,87.5,12.5,12.0,6.0,118.0,94.5
4,validation,ko,356,94.7,5.3,14.0,4.0,112.5,97.0
5,validation,te,384,75.8,24.2,19.5,6.0,142.0,88.2
